In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Separating dependant variables
data = load_breast_cancer()
X=data.data
y=data.target

In [ ]:
print(X.shape)

In [ ]:
# Pipe because first perform the Standard scalar then do logistic regression
model = Pipeline([
    ("Scalar",StandardScaler()),
    ("classifier",LogisticRegression(max_iter=5000))
])

selector = SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction="forward",
    scoring="accuracy",
    cv=5
)

selector.fit(X,y)

print(data.feature_names[selector.get_support()])

In [ ]:
# Backward Feature Selection
# selector = SequentialFeatureSelector(
#     model,
#     n_features_to_select=5,
#     direction="backward",
#     scoring="accuracy",
#     cv=5
# )

# selector.fit(X,y)

# print(data.feature_names[selector.get_support()])

In [ ]:
# Recursive Feature Elimination

from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X=data.data
y=data.target

X_scaled = StandardScaler().fit_transform(X)

model = LogisticRegression(max_iter=5000)

selector = RFE(
    estimator=model,
    n_features_to_select=5,
    step=1 # Remove 1 feature in each iteration.
)

selector.fit(X_scaled,y)

for name, selected, rank in zip(

    data.feature_names,
    selector.support_, # Boolean array showing each selected feature
    selector.ranking_ # RFE rank of every feature
):
    print(name,selected,rank)

In [ ]:
# RFECV

from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold # Beause using imbalanced dataset, not using regular K Fold

model = LogisticRegression(max_iter=5000)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state= 42
)

selector = RFECV(
    estimator=model,
    step =1,
    cv =cv,
    scoring="accuracy",
    min_features_to_select=1
)

selector.fit(X_scaled,y)

print("Optimal number of features:", selector.n_features_)
print("Selected features:", data.feature_names[selector.support_])

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

data = load_diabetes()
X=pd.DataFrame(data.data,columns=data.feature_names)
y= data.target

scalar = StandardScaler()
X_scaled = scalar.fit_transform(X)

lasso = LassoCV(
    cv =5,
    random_state=42,
    max_iter=10000
)

lasso.fit(X_scaled,y)

coefficients = pd.Series(
    lasso.coef_,
    index=X.columns
)
selected_features = coefficients[
    coefficients.abs() > 1e-6
]

print("Best alpha:",lasso.alpha_)
print("Selected features:")
print(selected_features)

In [ ]:
# L1-regularized Logistic Regression

from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data
y= data.target

pipeline = Pipeline([
    ("scalar",StandardScaler()),
    ("selection",SelectFromModel(
        LogisticRegression(
            # penalty="l1",
            l1_ratio=1,
            solver="liblinear", # Need to add to avoid error
            C=0.1, # Regularization Strength
            max_iter=5000
        ))),("classification",LogisticRegression(max_iter=5000))])

pipeline.fit(X,y)

mask = pipeline.named_steps["selection"].get_support()

print(data.feature_names[mask])


In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectFromModel

iris = load_iris()
X,y = iris.data,iris.target
X.shape

(150, 4)

In [30]:
clf = RandomForestClassifier(n_estimators=10)
# clf.feature_importances_

selector = SelectFromModel(estimator=clf, max_features=2)
selector.fit(X, y)

# Get the names of the selected features
selected_features = np.array(iris.feature_names)[selector.get_support()]

print("Selected features:", selected_features)

Selected features: ['petal length (cm)' 'petal width (cm)']


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
clfs = Pipeline([('feature_selection',SelectFromModel(RandomForestClassifier()))])